In [12]:
"""Modelagem da Avaliação A2 - classificação Normal x Anormal na base UCMF.


Protocolo decidido pelo grupo em 16/11/2026:
  - vazamento: excluídos os achados do exame físico (SOPRO, B2 e PULSOS), anotados no mesmo exame
    que produz o rótulo. O motivo do encaminhamento fica, porque é escrito antes da consulta;
  - divisão: 80/20 estratificada pelo alvo, sem agrupamento por `grupo_paciente` (a chave da A1 não
    identifica paciente - ver `preprocessamento.verificar_grupo_paciente`);
  - escolhas (hiperparâmetros, modelo e limiar): validação cruzada de 5 dobras dentro do treino,
    com o pré-processador reajustado em cada dobra;
  - o teste é aberto uma única vez, na Parte 4;
  - random_state = 42 em tudo.

"""

'Modelagem da Avaliação A2 - classificação Normal x Anormal na base UCMF.\n\n\nProtocolo decidido pelo grupo em 16/11/2026:\n  - vazamento: excluídos os achados do exame físico (SOPRO, B2 e PULSOS), anotados no mesmo exame\n    que produz o rótulo. O motivo do encaminhamento fica, porque é escrito antes da consulta;\n  - divisão: 80/20 estratificada pelo alvo, sem agrupamento por `grupo_paciente` (a chave da A1 não\n    identifica paciente - ver `preprocessamento.verificar_grupo_paciente`);\n  - escolhas (hiperparâmetros, modelo e limiar): validação cruzada de 5 dobras dentro do treino,\n    com o pré-processador reajustado em cada dobra;\n  - o teste é aberto uma única vez, na Parte 4;\n  - random_state = 42 em tudo.\n\n'

## 0. Configuração

In [13]:
import os
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use('Agg')          # sem interface gráfica: roda como script e no nbconvert

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter, PercentFormatter
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import partial_dependence, permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
                             make_scorer, precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import (GridSearchCV, StratifiedGroupKFold, StratifiedKFold,
                                     cross_val_predict, cross_validate)
from sklearn.pipeline import Pipeline

RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'UCMF_raw.csv').exists())
PASTA = RAIZ / 'Modelagem_A2'
FIG, TAB = PASTA / 'figuras', PASTA / 'tabelas'
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

# preprocessamento.py fica nesta pasta. Entra no sys.path (este processo) e no PYTHONPATH
# (processos paralelos que o scikit-learn abre quando n_jobs > 1).
if str(PASTA) not in sys.path:
    sys.path.insert(0, str(PASTA))
if str(PASTA) not in os.environ.get('PYTHONPATH', '').split(os.pathsep):
    os.environ['PYTHONPATH'] = os.pathsep.join(filter(None, [str(PASTA), os.environ.get('PYTHONPATH')]))

from preprocessamento import (CENARIO_PRINCIPAL, CENARIOS, FAIXAS_IDADE, RANDOM_STATE,
                              ROTULOS_FAIXAS, SEM_MOTIVO, TODOS, VAZAMENTO, PreprocessadorUCMF,
                              carregar_base, dividir_treino_teste, dobras, grupo_da_coluna,
                              verificar_grupo_paciente)

T0 = time.time()


def etapa(titulo):
    print(f'\n[{time.time() - T0:6.1f}s] {titulo}', flush=True)


# Mesmo estilo das figuras da A1: azul = Normal, laranja = Anormal
COR = {'Normal': '#2a78d6', 'Anormal': '#eb6834'}
COR_MODELO = {'Regressão Logística': '#008300', 'Random Forest': '#e87ba4', 'Gradient Boosting': '#6250d6'}
TINTA, TINTA_2, GRADE, EIXO = '#0b0b0b', '#52514e', '#e1e0d9', '#c3c2b7'
plt.rcParams.update({
    'text.color': TINTA, 'axes.labelcolor': TINTA, 'axes.titlecolor': TINTA,
    'axes.edgecolor': EIXO, 'xtick.color': TINTA_2, 'ytick.color': TINTA_2,
    'axes.grid': True, 'grid.color': GRADE, 'grid.linewidth': 0.5, 'axes.axisbelow': True,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white',
})

# Figuras no formato do relatório IEEE: largura de uma coluna ou da página inteira, 600 dpi,
# fontes de 7 a 8 pt no tamanho final, vírgula decimal e sem título (o título vai na legenda).
COLUNA, PAGINA = 3.5, 7.16
ESTILO_IEEE = {'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 7,
               'ytick.labelsize': 7, 'legend.fontsize': 7, 'axes.linewidth': 0.6,
               'grid.linewidth': 0.5, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
               'xtick.major.size': 2.5, 'ytick.major.size': 2.5}
PCT = PercentFormatter(1, decimals=0)


def num(x, casas=1):
    """Número com vírgula decimal (0,770)."""
    return f'{x:.{casas}f}'.replace('.', ',')


def fmt_n(n):
    """Inteiro com ponto como separador de milhar (4.144)."""
    return f'{int(n):,}'.replace(',', '.')


def virgula(casas=1):
    """Rótulos de eixo com vírgula decimal e sinal de menos tipográfico."""
    return FuncFormatter(lambda v, _: num(v, casas).replace('-', '−'))


def salvar(fig, nome):
    fig.savefig(FIG / f'{nome}.png', dpi=600, bbox_inches='tight')
    plt.close(fig)


def exportar(tabela, nome, index=False):
    tabela.to_csv(TAB / f'{nome}.csv', index=index, encoding='utf-8-sig')


print('pasta do projeto:', RAIZ.name, '| random_state =', RANDOM_STATE)

pasta do projeto: mineracao_de_dados | random_state = 42


## 1. Ponto de partida: base, divisão e pré-processador

Quatro verificações antes de qualquer modelo: a base é a da A1; o pré-processador reescrito
reproduz as matrizes da A1; a chave `grupo_paciente` não identifica paciente; e a exclusão do
exame físico retira as colunas certas, e só elas.

In [14]:
etapa('Parte 1 - base, divisão e verificações')

df, y = carregar_base(RAIZ)
idx_tr, idx_te = dividir_treino_teste(df, y)

X_treino, X_teste = df.iloc[idx_tr], df.iloc[idx_te]       # base limpa, ainda sem transformar
y_treino, y_teste = y.iloc[idx_tr], y.iloc[idx_te]
BASE = y.mean()                                            # 42,4% de Anormais na base limpa

resumo_divisao = pd.DataFrame({
    'consultas': [len(X_treino), len(X_teste)],
    '% Anormal': [round(y_treino.mean() * 100, 1), round(y_teste.mean() * 100, 1)],
}, index=['treino', 'teste'])
print(resumo_divisao.to_string())


[   0.0s] Parte 1 - base, divisão e verificações
        consultas  % Anormal
treino       8460       42.4
teste        2116       42.4


### 1.1 A chave `grupo_paciente` da A1 não identifica paciente

A A1 juntava consultas com a mesma data de atendimento e a mesma data de nascimento, supondo o
mesmo paciente, e usaria isso para agrupar a divisão. A verificação abaixo mostra que a suposição
não se sustenta, e por isso a divisão desta parte é estratificada simples.

In [15]:
ver_grupo = verificar_grupo_paciente(df)
print(ver_grupo.to_string(index=False))
exportar(ver_grupo, 'verificacao_grupo_paciente')

                              verificação  valor
                  consultas na base limpa  10576
      valores distintos de grupo_paciente  10407
          grupos com mais de uma consulta    160
                  consultas nesses grupos    329
                  maior grupo (consultas)      4
 grupos multi-consulta com SEXO diferente     31
grupos multi-consulta com laudo diferente     42


### 1.2 O pré-processador reescrito reproduz a A1?

Com `excluir_grupos=()` ele gera as 47 colunas da A1. Para conferir, a divisão da A1 é
reconstruída aqui - e SÓ aqui, para esta verificação: o pré-processador, ajustado naquele mesmo
treino, tem de reproduzir os arquivos `treino.csv` e `teste.csv` exportados pela bivariada. A
modelagem desta parte usa a divisão estratificada simples definida em 1.

In [16]:
PASTA_A1 = RAIZ / 'Análise Bivariada' / 'modelagem'
split_a1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
i_tr_a1, i_te_a1 = next(split_a1.split(df, y, groups=df['grupo_paciente']))
prep_a1 = PreprocessadorUCMF(excluir_grupos=TODOS).fit(df.iloc[i_tr_a1])
for nome, idx in [('treino', i_tr_a1), ('teste', i_te_a1)]:
    ref = pd.read_csv(PASTA_A1 / f'{nome}.csv')
    ref_X = ref.drop(columns=['alvo', 'grupo_paciente'])
    novo = prep_a1.transform(df.iloc[idx])
    assert list(novo.columns) == list(ref_X.columns), f'colunas diferentes em {nome}'
    assert np.allclose(novo.values, ref_X.values, rtol=0, atol=1e-9), f'valores diferentes em {nome}'
    print(f'{nome}: {novo.shape} igual ao {nome}.csv da A1 '
          f'(maior diferença {np.abs(novo.values - ref_X.values).max():.0e}, arredondamento do CSV)')

treino: (8460, 47) igual ao treino.csv da A1 (maior diferença 2e-15, arredondamento do CSV)
teste: (2116, 47) igual ao teste.csv da A1 (maior diferença 9e-16, arredondamento do CSV)


### 1.3 Exclusão dos achados do exame físico (vazamento de informação)

O rótulo Normal/Anormal é a conclusão do cardiologista depois do exame. SOPRO, B2 e PULSOS são o
que ele ouviu e palpou nesse mesmo exame: usá-los é prever o laudo com o próprio laudo. Os três
saem juntos porque o critério é a origem da informação, não a força da associação.

In [17]:
prep = PreprocessadorUCMF().fit(X_treino)                  # padrão: exclui achados_exame_fisico
cols_a1 = list(prep_a1.get_feature_names_out())
cols_p = list(prep.get_feature_names_out())
excluidas = [c for c in cols_a1 if c not in set(cols_p)]
print(f'{len(cols_a1)} colunas -> {len(cols_p)} colunas | excluídas ({len(excluidas)}): {excluidas}')

por_grupo = pd.DataFrame({
    'A1 (todas)': pd.Series(cols_a1).map(grupo_da_coluna).value_counts(),
    'A2 (sem exame físico)': pd.Series(cols_p).map(grupo_da_coluna).value_counts(),
}).fillna(0).astype(int)
por_grupo.loc['total'] = por_grupo.sum()
print(por_grupo.to_string())
exportar(por_grupo.reset_index(names='grupo de atributos'), 'colunas_por_grupo')

47 colunas -> 37 colunas | excluídas (10): ['pulsos_alterados', 'SOPRO_Ausente', 'SOPRO_Sistólico', 'SOPRO_Contínuo', 'SOPRO_raras', 'B2_Normal', 'B2_Hiperfonética', 'B2_Desdobramento fixo', 'B2_Outro', 'B2_Única']
                         A1 (todas)  A2 (sem exame físico)
achados_exame_fisico             10                      0
antropometria                     2                      2
demografica                       2                      2
motivo_consulta                  18                     18
sinais_vitais_e_pressao           6                      6
sintomas                          9                      9
total                            47                     37


### 1.4 O tamanho do vazamento, na base limpa inteira

In [18]:
achados = [('Sopro sistólico', df['SOPRO'] == 'Sistólico'),
           ('Sopro contínuo', df['SOPRO'] == 'Contínuo'),
           ('B2 hiperfonética', df['B2'] == 'Hiperfonética'),
           ('B2 única', df['B2'] == 'Única'),
           ('B2 com desdobramento fixo', df['B2'] == 'Desdobramento fixo'),
           ('B2 com outra alteração', df['B2'] == 'Outro'),
           ('Pulsos alterados', df['PULSOS'].notna() & (df['PULSOS'] != 'Normais'))]
alterado = ((df['SOPRO'].fillna('Ausente') != 'Ausente') | (df['B2'].fillna('Normal') != 'Normal')
            | (df['PULSOS'].fillna('Normais') != 'Normais'))
achados.append(('Nenhum achado alterado', ~alterado))

evidencia = pd.DataFrame([{'achado do exame físico': r, 'consultas': int(m.sum()),
                           '% Anormal': round(y[m].mean() * 100, 1)} for r, m in achados])
print(evidencia.to_string(index=False))
# Uma consulta pode ter mais de um achado alterado, então as linhas acima não somam: o total vem
# da união das três condições.
print(f'algum achado alterado: {fmt_n(alterado.sum())} consultas, {y[alterado].mean() * 100:.1f}% Anormais | '
      f'nenhum: {fmt_n((~alterado).sum())}, {y[~alterado].mean() * 100:.1f}%')
exportar(evidencia, 'vazamento_exame_fisico')

# Alvo por faixa etária e por motivo, na base limpa inteira: são os números citados na seção de
# Dados do relatório, e os mesmos que a Fig. 2 desenha.
COLS_MOTIVO1 = {'motivo_parecer_cardiologico': 'Parecer cardiológico',
                'motivo_suspeita_de_cardiopatia': 'Suspeita de cardiopatia',
                'motivo_cardiopatia_ja_estabelecida': 'Cardiopatia já estabelecida',
                'motivo_check_up': 'Check-up', 'motivo_outro': 'Outro'}
faixa_base = pd.cut(df['IDADE'], FAIXAS_IDADE, labels=ROTULOS_FAIXAS, include_lowest=True)
alvo_faixa = (y.groupby(faixa_base, observed=True).agg(consultas='size', pct_anormal='mean')
              .assign(pct_anormal=lambda d: (d['pct_anormal'] * 100).round(1)))
marc_base = df[list(COLS_MOTIVO1)]
motivo1_base = (marc_base.idxmax(axis=1).map(COLS_MOTIVO1)
                .where(marc_base.sum(axis=1) == 1, 'Não informado').rename('motivo principal'))
alvo_motivo = (y.groupby(motivo1_base).agg(consultas='size', pct_anormal='mean')
               .assign(pct_anormal=lambda d: (d['pct_anormal'] * 100).round(1))
               .sort_values('pct_anormal', ascending=False))
print(alvo_faixa.to_string())
print(alvo_motivo.to_string())
exportar(alvo_faixa.reset_index(names='faixa etária'), 'alvo_por_faixa')
exportar(alvo_motivo.reset_index(), 'alvo_por_motivo')

   achado do exame físico  consultas  % Anormal
          Sopro sistólico       3948       95.9
           Sopro contínuo         37       75.7
         B2 hiperfonética        248       98.0
                 B2 única         65       96.9
B2 com desdobramento fixo        136       90.4
   B2 com outra alteração         65       86.2
         Pulsos alterados        122       85.2
   Nenhum achado alterado       6432        8.6
algum achado alterado: 4.144 consultas, 95.0% Anormais | nenhum: 6.432, 8.6%
       consultas  pct_anormal
IDADE                        
0–1         2639         61.2
1–5         3517         40.2
5–10        2584         35.8
10–15       1574         29.2
15–19        262         27.9
                             consultas  pct_anormal
motivo principal                                   
Cardiopatia já estabelecida        986         71.2
Suspeita de cardiopatia           3804         58.9
Não informado                      100         50.0
Outro                

## 2. Protocolo de validação e modelos de referência

O teste fica guardado até a Parte 4. Toda escolha é feita com validação cruzada de 5 dobras
**dentro do treino**, estratificada pelo alvo. O pré-processador está dentro do `Pipeline`, então
é reajustado em cada dobra, só com as linhas de ajuste daquela dobra.

In [19]:
etapa('Parte 2 - dobras, Dummy e regressão logística')

cv = dobras()
METRICAS = {
    'AUC': 'roc_auc',
    'sensibilidade': make_scorer(recall_score, zero_division=0),
    'especificidade': make_scorer(recall_score, pos_label=0, zero_division=0),
    'precisão': make_scorer(precision_score, zero_division=0),
    'F1': make_scorer(f1_score, zero_division=0),
    'acurácia balanceada': 'balanced_accuracy',
    'acurácia': 'accuracy',
}
resultados_cv = {}   # modelo -> métricas em cada dobra (5 linhas x 7 métricas)
oof = {}             # modelo -> probabilidade out-of-fold de Anormal, uma por linha do treino


def pipeline(modelo, **kw_prep):
    """Pré-processador (reajustado em cada dobra) + modelo."""
    return Pipeline([('prep', PreprocessadorUCMF(**kw_prep)), ('modelo', modelo)])


def avaliar_cv(nome, modelo):
    r = cross_validate(modelo, X_treino, y_treino, cv=cv, scoring=METRICAS, n_jobs=-1)
    resultados_cv[nome] = pd.DataFrame({m: r[f'test_{m}'] for m in METRICAS})
    oof[nome] = cross_val_predict(modelo, X_treino, y_treino, cv=cv, method='predict_proba',
                                  n_jobs=-1)[:, 1]


def tabela_cv(nomes=None):
    """Média ± desvio-padrão das 5 dobras, por modelo."""
    nomes = nomes or list(resultados_cv)
    return pd.DataFrame({n: {m: f'{num(v.mean(), 3)} ± {num(v.std(), 3)}'
                             for m, v in resultados_cv[n].items()} for n in nomes}).T


def resumo_busca(busca, n=5):
    """As n melhores combinações da grade, com a AUC no ajuste e na validação."""
    r = pd.DataFrame(busca.cv_results_)
    params = [c for c in r if c.startswith('param_')]
    t = r[params + ['mean_train_score', 'mean_test_score', 'std_test_score', 'rank_test_score']].copy()
    t.columns = ([c.replace('param_modelo__', '') for c in params]
                 + ['AUC ajuste', 'AUC validação', 'desvio', 'posição'])
    print(f'{len(t)} combinações | AUC de validação entre {num(t["AUC validação"].min(), 3)} '
          f'e {num(t["AUC validação"].max(), 3)}')
    return t.sort_values('posição').head(n).round(4)


# Regressão logística: baseline simples e interpretável. Único hiperparâmetro é C, o inverso da
# força da regularização L2. Sem pesos de classe: o desbalanceamento é leve (57,6% x 42,4%) e o
# equilíbrio entre sensibilidade e especificidade é tratado pelo limiar (Parte 4).
busca_log = GridSearchCV(pipeline(LogisticRegression(max_iter=5000)),
                         {'modelo__C': [0.001, 0.01, 0.1, 1, 10, 100]},
                         scoring='roc_auc', cv=cv, n_jobs=-1, return_train_score=True)
busca_log.fit(X_treino, y_treino)
grade_log = resumo_busca(busca_log, n=6)
print(grade_log.to_string(index=False))
print('C escolhido:', busca_log.best_params_['modelo__C'])
exportar(grade_log, 'grade_logistica')

avaliar_cv('Dummy', pipeline(DummyClassifier(strategy='most_frequent')))
avaliar_cv('Regressão Logística', busca_log.best_estimator_)
print(tabela_cv().to_string())


[   5.4s] Parte 2 - dobras, Dummy e regressão logística
6 combinações | AUC de validação entre 0,742 e 0,764
      C  AUC ajuste  AUC validação  desvio  posição
  0.100      0.7698         0.7643  0.0075        1
  1.000      0.7706         0.7640  0.0084        2
 10.000      0.7707         0.7640  0.0087        3
100.000      0.7707         0.7639  0.0088        4
  0.010      0.7641         0.7604  0.0063        5
  0.001      0.7445         0.7423  0.0076        6
C escolhido: 0.1
                               AUC  sensibilidade especificidade       precisão             F1 acurácia balanceada       acurácia
Dummy                0,500 ± 0,000  0,000 ± 0,000  1,000 ± 0,000  0,000 ± 0,000  0,000 ± 0,000       0,500 ± 0,000  0,576 ± 0,000
Regressão Logística  0,764 ± 0,008  0,566 ± 0,016  0,840 ± 0,011  0,722 ± 0,017  0,635 ± 0,016       0,703 ± 0,012  0,724 ± 0,012


## 3. Modelos complexos: Random Forest e Gradient Boosting

Os dois combinam árvores de decisão, de formas opostas. Random Forest (*bagging*): centenas de
árvores profundas, cada uma numa amostra *bootstrap* das linhas e olhando um sorteio de colunas a
cada divisão; a média reduz a variância. Gradient Boosting (*boosting*): árvores pequenas em
sequência, cada uma corrigindo o erro acumulado das anteriores; a sequência reduz o viés. Os dois
captam o que a logística não capta: relações não lineares e interações entre atributos.

In [9]:
etapa('Parte 3 - Random Forest')

busca_rf = GridSearchCV(
    pipeline(RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)),
    {'modelo__min_samples_leaf': [1, 5, 10, 20, 40],
     'modelo__max_features': ['sqrt', 0.33, 0.5]},
    scoring='roc_auc', cv=cv, n_jobs=-1, return_train_score=True)
busca_rf.fit(X_treino, y_treino)
grade_rf = resumo_busca(busca_rf)
print(grade_rf.to_string(index=False))
exportar(grade_rf, 'grade_random_forest')

etapa('Parte 3 - Gradient Boosting')
busca_gb = GridSearchCV(
    pipeline(HistGradientBoostingClassifier(early_stopping=False, random_state=RANDOM_STATE)),
    {'modelo__learning_rate': [0.02, 0.05, 0.1],
     'modelo__max_iter': [100, 300],
     'modelo__max_leaf_nodes': [7, 15, 31],
     'modelo__min_samples_leaf': [20, 50]},
    scoring='roc_auc', cv=cv, n_jobs=-1, return_train_score=True)
busca_gb.fit(X_treino, y_treino)
grade_gb = resumo_busca(busca_gb)
print(grade_gb.to_string(index=False))
exportar(grade_gb, 'grade_gradient_boosting')

BUSCAS = {'Regressão Logística': busca_log, 'Random Forest': busca_rf, 'Gradient Boosting': busca_gb}
hiper = pd.DataFrame([{'modelo': n,
                       'hiperparâmetros escolhidos': ', '.join(
                           f'{k.replace("modelo__", "")} = {v}' for k, v in b.best_params_.items()),
                       'combinações na grade': len(b.cv_results_['params']),
                       'AUC média na validação cruzada': round(b.best_score_, 4)}
                      for n, b in BUSCAS.items()])
print(hiper.to_string(index=False))
exportar(hiper, 'hiperparametros')


[  41.8s] Parte 3 - Random Forest
15 combinações | AUC de validação entre 0,754 e 0,775
max_features  min_samples_leaf  AUC ajuste  AUC validação  desvio  posição
        sqrt                 5      0.8880         0.7750  0.0115        1
        sqrt                10      0.8444         0.7744  0.0104        2
        0.33                10      0.8739         0.7743  0.0108        3
        0.33                20      0.8339         0.7739  0.0104        4
         0.5                20      0.8428         0.7736  0.0098        5

[ 200.6s] Parte 3 - Gradient Boosting
36 combinações | AUC de validação entre 0,755 e 0,777
 learning_rate  max_iter  max_leaf_nodes  min_samples_leaf  AUC ajuste  AUC validação  desvio  posição
          0.05       100              31                20      0.8607         0.7766  0.0133        1
          0.02       300              15                20      0.8291         0.7764  0.0135        2
          0.05       100              31                50 

### 3.1 Comparação dos quatro modelos na validação cruzada

In [10]:
etapa('Parte 3.1 - comparação na validação cruzada')

avaliar_cv('Random Forest', busca_rf.best_estimator_)
avaliar_cv('Gradient Boosting', busca_gb.best_estimator_)
comparativa_cv = tabela_cv()
print(comparativa_cv.to_string())
exportar(comparativa_cv.reset_index(names='modelo'), 'cv_comparativa')
exportar(pd.concat({n: r for n, r in resultados_cv.items()}, names=['modelo', 'dobra']).reset_index(),
         'cv_por_dobra')

# As quatro avaliações usaram as MESMAS 5 dobras, então dá para comparar dobra a dobra: uma
# vantagem real aparece em todas as dobras, uma vantagem de acaso muda de sinal.
MODELOS = ['Regressão Logística', 'Random Forest', 'Gradient Boosting']
auc_dobras = pd.DataFrame({n: resultados_cv[n]['AUC'] for n in MODELOS})
auc_dobras.index = [f'dobra {k}' for k in range(1, 6)]
ganho = auc_dobras[['Random Forest', 'Gradient Boosting']].sub(auc_dobras['Regressão Logística'], axis=0)
ganho.columns = [f'{c} − Logística' for c in ganho.columns]
dobra_a_dobra = auc_dobras.join(ganho).round(4)
print(dobra_a_dobra.to_string())
exportar(dobra_a_dobra.reset_index(names='dobra'), 'cv_dobra_a_dobra')


[ 291.9s] Parte 3.1 - comparação na validação cruzada
                               AUC  sensibilidade especificidade       precisão             F1 acurácia balanceada       acurácia
Dummy                0,500 ± 0,000  0,000 ± 0,000  1,000 ± 0,000  0,000 ± 0,000  0,000 ± 0,000       0,500 ± 0,000  0,576 ± 0,000
Regressão Logística  0,764 ± 0,008  0,566 ± 0,016  0,840 ± 0,011  0,722 ± 0,017  0,635 ± 0,016       0,703 ± 0,012  0,724 ± 0,012
Random Forest        0,775 ± 0,013  0,566 ± 0,015  0,849 ± 0,007  0,734 ± 0,012  0,639 ± 0,013       0,708 ± 0,010  0,729 ± 0,009
Gradient Boosting    0,777 ± 0,015  0,583 ± 0,012  0,835 ± 0,011  0,722 ± 0,016  0,645 ± 0,013       0,709 ± 0,010  0,728 ± 0,010
         Regressão Logística  Random Forest  Gradient Boosting  Random Forest − Logística  Gradient Boosting − Logística
dobra 1               0.7584         0.7677             0.7637                     0.0093                         0.0053
dobra 2               0.7620         0.7738          

## 4. Limiar de decisão e avaliação final no teste

Com o limiar padrão (0,5) o modelo favorece a classe mais frequente e deixa passar perto de
metade dos Anormais - inútil para uma triagem, em que deixar passar um cardiopata é pior que um
falso alarme. O limiar é escolhido para uma sensibilidade-alvo de 0,80, **só** com as previsões
*out-of-fold* do treino, e fixado antes de olhar o teste.

In [11]:
etapa('Parte 4 - limiar e teste')

SENS_ALVO = 0.80


def limiar_para_sensibilidade(y_true, p, alvo):
    """Maior limiar cuja sensibilidade chega ao alvo."""
    _, tpr, lim = roc_curve(y_true, p)
    return lim[np.argmax(tpr >= alvo)]


def metricas_no_limiar(y_true, p, limiar):
    prev = (p >= limiar).astype(int)
    return {'limiar': limiar,
            'sensibilidade': recall_score(y_true, prev, zero_division=0),
            'especificidade': recall_score(y_true, prev, pos_label=0, zero_division=0),
            'precisão': precision_score(y_true, prev, zero_division=0),
            '% priorizados': prev.mean() * 100}


p_log = oof['Regressão Logística']
linhas = {'padrão (0,5)': metricas_no_limiar(y_treino, p_log, 0.5)}
for alvo in [0.75, 0.80, 0.85, 0.90]:
    linhas[f'sensibilidade >= {num(alvo, 2)}'] = metricas_no_limiar(
        y_treino, p_log, limiar_para_sensibilidade(y_treino, p_log, alvo))
limiares_oof = pd.DataFrame(linhas).T.astype(float).round(3)
print(limiares_oof.to_string())
exportar(limiares_oof.reset_index(names='critério'), 'limiar_oof')


[ 304.1s] Parte 4 - limiar e teste
                       limiar  sensibilidade  especificidade  precisão  % priorizados
padrão (0,5)            0.500          0.566           0.840     0.722         33.215
sensibilidade >= 0,75   0.323          0.750           0.635     0.602         52.837
sensibilidade >= 0,80   0.293          0.800           0.553     0.568         59.704
sensibilidade >= 0,85   0.267          0.850           0.455     0.535         67.459
sensibilidade >= 0,90   0.238          0.900           0.328     0.497         76.879


### 4.1 Tabela comparativa no teste

O teste (2.116 consultas) é usado aqui pela primeira e única vez. Cada modelo é o vencedor da sua
grade, já reajustado no treino inteiro pelo `GridSearchCV`. Os intervalos de confiança de 95% vêm
de 1.000 reamostragens *bootstrap* do teste; com cerca de 2 mil linhas, a AUC varia perto de
±0,02 só pelo sorteio de quem caiu no teste.

In [20]:
finais = {'Dummy': pipeline(DummyClassifier(strategy='most_frequent')).fit(X_treino, y_treino),
          **{n: b.best_estimator_ for n, b in BUSCAS.items()}}
p_teste = {n: m.predict_proba(X_teste)[:, 1] for n, m in finais.items()}
limiar = {n: limiar_para_sensibilidade(y_treino, oof[n], SENS_ALVO) for n in MODELOS}
print('limiares para sensibilidade 0,80 (escolhidos no treino):', {n: num(t, 3) for n, t in limiar.items()})

yt = y_teste.values


def metricas(p, lim):
    prev = (p >= lim).astype(int)
    return {'AUC': roc_auc_score(yt, p),
            'sensibilidade': recall_score(yt, prev, zero_division=0),
            'especificidade': recall_score(yt, prev, pos_label=0, zero_division=0),
            'precisão': precision_score(yt, prev, zero_division=0),
            'F1': f1_score(yt, prev, zero_division=0),
            'acurácia balanceada': balanced_accuracy_score(yt, prev),
            'acurácia': accuracy_score(yt, prev),
            '% priorizados': prev.mean() * 100}


# bootstrap por linha: cada consulta é uma observação independente (não há identificador de paciente)
N_BOOT = 1000
rng = np.random.default_rng(RANDOM_STATE)
amostras = [rng.integers(0, len(yt), len(yt)) for _ in range(N_BOOT)]


def ic95(valores):
    lo, hi = np.percentile(valores, [2.5, 97.5])
    return f'{num(lo, 3)} a {num(hi, 3)}'


auc_boot = pd.DataFrame({n: [roc_auc_score(yt[a], p_teste[n][a]) for a in amostras] for n in MODELOS})

linhas = {}
for n in ['Dummy'] + MODELOS:
    linha = {'AUC validação cruzada (média ± dp)': f'{num(resultados_cv[n]["AUC"].mean(), 3)} ± '
                                                   f'{num(resultados_cv[n]["AUC"].std(), 3)}',
             'AUC teste': num(roc_auc_score(yt, p_teste[n]), 3),
             'IC 95% (teste)': '—' if n == 'Dummy' else ic95(auc_boot[n])}
    if n == 'Dummy':
        linha.update({k: '—' for k in ['limiar', 'sensibilidade', 'especificidade', 'precisão', 'F1',
                                       'acurácia balanceada', '% priorizados']})
    else:
        m = metricas(p_teste[n], limiar[n])
        linha.update({'limiar': num(limiar[n], 2),
                      **{k: num(m[k], 3) for k in ['sensibilidade', 'especificidade', 'precisão',
                                                   'F1', 'acurácia balanceada']},
                      '% priorizados': num(m['% priorizados'])})
    linhas[n] = linha
tabela_comparativa = pd.DataFrame(linhas).T
print(tabela_comparativa.to_string())
exportar(tabela_comparativa.reset_index(names='modelo'), 'tabela_comparativa')

# limiar 0,5, para comparar com a validação cruzada
tab_05 = pd.DataFrame({n: metricas(p_teste[n], 0.5) for n in finais}).T.round(3)
exportar(tab_05.reset_index(names='modelo'), 'teste_limiar_05')

# diferenças de AUC entre modelos no MESMO sorteio de consultas (bootstrap pareado): é a única
# comparação que separa modelos cujos intervalos individuais se sobrepõem
pares = [('Gradient Boosting', 'Regressão Logística'), ('Random Forest', 'Regressão Logística'),
         ('Gradient Boosting', 'Random Forest')]
dif = pd.DataFrame({f'{a} − {b}': {
    'diferença de AUC': round(roc_auc_score(yt, p_teste[a]) - roc_auc_score(yt, p_teste[b]), 4),
    'IC 95%': ic95(auc_boot[a] - auc_boot[b]),
    '% das reamostragens > 0': round((auc_boot[a] > auc_boot[b]).mean() * 100, 1)}
    for a, b in pares}).T
print(dif.to_string())
exportar(dif.reset_index(names='comparação'), 'diferencas_auc')

KeyError: 'Random Forest'

### 4.2 Matriz de confusão e onde o modelo erra

In [ ]:
gb = 'Gradient Boosting'
prev_gb = p_teste[gb] >= limiar[gb]
cm = confusion_matrix(yt, prev_gb.astype(int))
pct = cm / cm.sum(axis=1, keepdims=True)
matriz = pd.DataFrame([[f'{fmt_n(cm[i, j])} ({num(pct[i, j] * 100)}%)' for j in range(2)] for i in range(2)],
                      index=['Real: Normal', 'Real: Anormal'],
                      columns=['Liberado (previsto Normal)', 'Priorizado (previsto Anormal)'])
print(matriz.to_string())
print(f'Anormais não encontrados: {fmt_n(cm[1, 0])} | Normais priorizados: {fmt_n(cm[0, 1])}')
exportar(matriz.reset_index(names=f'{gb}, limiar {num(limiar[gb], 2)}'), 'matriz_confusao')


def erros_por(grupo, p, lim):
    """Sensibilidade e especificidade dentro de cada categoria de `grupo`, no teste."""
    prev = (p >= lim).astype(int)
    d = pd.DataFrame({'grupo': grupo.values, 'y': yt, 'prev': prev})
    linhas = []
    for g_, s in d.groupby('grupo', observed=True, sort=False):
        pos, neg = s[s.y == 1], s[s.y == 0]
        linhas.append({grupo.name: g_, 'n': len(s), '% Anormal': round(s.y.mean() * 100, 1),
                       '% priorizados': round(s.prev.mean() * 100, 1),
                       'sensibilidade': round(pos.prev.mean(), 3) if len(pos) else np.nan,
                       'especificidade': round(1 - neg.prev.mean(), 3) if len(neg) else np.nan,
                       'Anormais não encontrados': int(((s.y == 1) & (s.prev == 0)).sum()),
                       'Normais priorizados': int(((s.y == 0) & (s.prev == 1)).sum())})
    return pd.DataFrame(linhas).set_index(grupo.name)


faixa_teste = pd.cut(X_teste['IDADE'], FAIXAS_IDADE, labels=ROTULOS_FAIXAS,
                     include_lowest=True).rename('faixa etária')
erros_faixa = erros_por(faixa_teste, p_teste[gb], limiar[gb]).reindex(ROTULOS_FAIXAS)
print(erros_faixa.to_string())
exportar(erros_faixa.reset_index(), 'erros_por_faixa')

# MOTIVO1 reconstruído das colunas binárias, como na bivariada da A1 (COLS_MOTIVO1 vem de 1.4)
marcados = X_teste[list(COLS_MOTIVO1)]
motivo1_teste = (marcados.idxmax(axis=1).map(COLS_MOTIVO1)
                 .where(marcados.sum(axis=1) == 1, 'Não informado').rename('motivo principal'))
erros_motivo = erros_por(motivo1_teste, p_teste[gb], limiar[gb]).sort_values('% Anormal', ascending=False)
print(erros_motivo.to_string())
exportar(erros_motivo.reset_index(), 'erros_por_motivo')

fn_total = int(cm[1, 0])
fn_parecer = int(erros_motivo.loc['Parecer cardiológico', 'Anormais não encontrados'])
print(f'dos {fn_total} Anormais não encontrados, {fn_parecer} ({fn_parecer / fn_total * 100:.0f}%) '
      'vieram para parecer cardiológico')

              Liberado (previsto Normal) Priorizado (previsto Anormal)
Real: Normal                 761 (62,4%)                   458 (37,6%)
Real: Anormal                170 (19,0%)                   727 (81,0%)
Anormais não encontrados: 170 | Normais priorizados: 458
                n  % Anormal  % priorizados  sensibilidade  especificidade  Anormais não encontrados  Normais priorizados
faixa etária                                                                                                             
0–1           543       60.0           85.3          0.963           0.313                        12                  149
1–5           696       41.8           54.9          0.804           0.635                        57                  148
5–10          515       35.3           45.2          0.670           0.667                        60                  111
10–15         310       27.4           31.9          0.600           0.787                        34                   4

## 5. Importância dos atributos

Importância por permutação: embaralha-se uma coluna (ou um grupo de colunas) no teste, o que
quebra sua relação com o alvo, e mede-se quanto a AUC cai. O método vale para qualquer modelo,
então compara os três. A permutação é feita também **por grupo**, porque as 18 colunas de motivo
descrevem uma só informação: embaralhadas uma a uma, uma compensa a outra e o peso do conjunto
fica subestimado. Não se usa a importância por impureza das árvores, que favorece atributos com
muitos valores distintos.

In [ ]:
etapa('Parte 5 - importância por grupo')

Xt_teste = {n: finais[n].named_steps['prep'].transform(X_teste) for n in MODELOS}
colunas = list(Xt_teste[gb].columns)
GRUPOS = {
    'Motivo do encaminhamento': [c for c in colunas if c.startswith('motivo_')],
    'Idade': ['IDADE_log_z'],
    'Sintomas (HDA)': [c for c in colunas if c.startswith('sint_')] + ['hda_ausente'],
    'Frequência cardíaca': ['FC_z'],
    'Pressão arterial': ['PAS_z', 'PAD_z', 'pa_medida', 'PPA_ord', 'ppa_calculada'],
    'IMC': ['IMC_z'],
    'Sexo': ['SEXO_M', 'sexo_ausente'],
}
assert sorted(sum(GRUPOS.values(), [])) == sorted(colunas), 'coluna sem grupo ou em dois grupos'
N_REP = 30


def importancia_grupos(modelo, Xt):
    """Queda na AUC ao embaralhar, juntas, as colunas de cada grupo (30 repetições)."""
    rng_ = np.random.default_rng(RANDOM_STATE)
    base = roc_auc_score(yt, modelo.predict_proba(Xt)[:, 1])
    quedas = {}
    for g_, cols in GRUPOS.items():
        quedas[g_] = []
        for _ in range(N_REP):
            Xp = Xt.copy()
            Xp[cols] = Xt[cols].values[rng_.permutation(len(Xt))]   # mesma permutação para o grupo
            quedas[g_].append(base - roc_auc_score(yt, modelo.predict_proba(Xp)[:, 1]))
    return pd.DataFrame(quedas)


imp_grupos = {n: importancia_grupos(finais[n].named_steps['modelo'], Xt_teste[n]) for n in MODELOS}
imp_media = pd.DataFrame({n: d.mean() for n, d in imp_grupos.items()})
imp_dp = pd.DataFrame({n: d.std() for n, d in imp_grupos.items()})
ordem_grupos = imp_media[gb].sort_values(ascending=False).index

tab_grupos = pd.DataFrame({n: [f'{num(imp_media.loc[g_, n], 3)} ± {num(imp_dp.loc[g_, n], 3)}'
                               for g_ in ordem_grupos] for n in MODELOS}, index=ordem_grupos)
tab_grupos.insert(0, 'colunas', [len(GRUPOS[g_]) for g_ in ordem_grupos])
print(tab_grupos.to_string())
exportar(imp_media.loc[ordem_grupos].add_suffix(' (queda média de AUC)')
         .join(imp_dp.loc[ordem_grupos].add_suffix(' (desvio)')).reset_index(names='grupo'),
         'importancia_grupos')


[ 168.9s] Parte 5 - importância por grupo
                          colunas Regressão Logística  Random Forest Gradient Boosting
Motivo do encaminhamento       18       0,212 ± 0,008  0,217 ± 0,009     0,204 ± 0,009
Idade                           1       0,022 ± 0,004  0,014 ± 0,003     0,028 ± 0,004
Frequência cardíaca             1       0,012 ± 0,003  0,022 ± 0,005     0,027 ± 0,005
Sintomas (HDA)                  9       0,024 ± 0,003  0,021 ± 0,003     0,022 ± 0,004
Pressão arterial                5       0,009 ± 0,002  0,010 ± 0,003     0,008 ± 0,002
IMC                             1       0,005 ± 0,002  0,003 ± 0,002     0,005 ± 0,002
Sexo                            2      -0,000 ± 0,001  0,003 ± 0,001     0,004 ± 0,001


### 5.1 Atributos individuais e direção dos efeitos

In [ ]:
etapa('Parte 5.1 - importância por coluna e odds ratios')

imp_ind = {}
for n in [gb, 'Regressão Logística']:
    r = permutation_importance(finais[n].named_steps['modelo'], Xt_teste[n], yt, scoring='roc_auc',
                               n_repeats=N_REP, random_state=RANDOM_STATE, n_jobs=-1)
    imp_ind[n] = pd.DataFrame({'média': r.importances_mean, 'desvio': r.importances_std}, index=colunas)

grupo_de = {c: g_ for g_, cols in GRUPOS.items() for c in cols}
tab_ind = pd.DataFrame({'grupo': [grupo_de[c] for c in colunas],
                        gb: imp_ind[gb]['média'], 'GB (desvio)': imp_ind[gb]['desvio'],
                        'Regressão Logística': imp_ind['Regressão Logística']['média']},
                       index=colunas).sort_values(gb, ascending=False)
tab_ind['posição na logística'] = tab_ind['Regressão Logística'].rank(ascending=False).astype(int)
print(tab_ind.head(12).round(4).to_string())
exportar(tab_ind.round(5).reset_index(names='coluna'), 'importancia_atributos')

# odds ratio: quanto a chance de exame anormal se multiplica quando o atributo aumenta. Nas colunas
# binárias, ter (1) contra não ter (0); nas numéricas, em z-score, o efeito de 1 desvio-padrão.
prep_final = finais['Regressão Logística'].named_steps['prep']
coef = pd.Series(finais['Regressão Logística'].named_steps['modelo'].coef_[0], index=colunas)
Xt_tr = prep_final.transform(X_treino)
binaria = Xt_tr.isin([0, 1]).all()
ors = pd.DataFrame({'grupo': [grupo_de[c] for c in colunas],
                    'odds ratio': np.exp(coef).round(3),
                    'casos no treino': [int(Xt_tr[c].sum()) if binaria[c] else None for c in colunas]},
                   index=colunas).sort_values('odds ratio', ascending=False)
ors['casos no treino'] = ors['casos no treino'].astype('Int64')
print(pd.concat([ors.head(6), ors.tail(6)]).to_string())
exportar(ors.reset_index(names='coluna'), 'odds_ratios')


[ 200.2s] Parte 5.1 - importância por coluna e odds ratios
                                                       grupo  Gradient Boosting  GB (desvio)  Regressão Logística  posição na logística
motivo_sopro                        Motivo do encaminhamento             0.0826       0.0062               0.0557                     1
FC_z                                     Frequência cardíaca             0.0279       0.0048               0.0112                     5
IDADE_log_z                                            Idade             0.0256       0.0050               0.0212                     3
motivo_cardiopatia_ja_estabelecida  Motivo do encaminhamento             0.0174       0.0025               0.0091                     6
motivo_parecer_cardiologico         Motivo do encaminhamento             0.0137       0.0029               0.0241                     2
hda_ausente                                   Sintomas (HDA)             0.0118       0.0023               0.0123           

### 5.2 O encaminhamento por sopro não é o sopro do exame

`motivo_sopro` é a coluna mais importante dos modelos, e é razoável perguntar se ela não é o
atributo excluído entrando por outra porta. A tabela abaixo mostra que não: o encaminhamento é a
suspeita do pediatra, escrita antes da consulta, e erra em uma de cada cinco crianças.

In [ ]:
enc_sopro = df['motivo_sopro'] == 1
sopro_exame = df['SOPRO'].fillna('Ausente') != 'Ausente'
tab_sopro = (pd.DataFrame({'encaminhado por sopro': np.where(enc_sopro, 'sim', 'não'),
                           'sopro no exame': np.where(sopro_exame, 'sim', 'não'), 'Anormal': y})
             .groupby(['encaminhado por sopro', 'sopro no exame'])['Anormal'].agg(['size', 'mean']))
tab_sopro.columns = ['consultas', '% Anormal']
tab_sopro['% Anormal'] = (tab_sopro['% Anormal'] * 100).round(1)
print(tab_sopro.to_string())
print(f'encaminhados por sopro: {fmt_n(enc_sopro.sum())} ({enc_sopro.mean() * 100:.1f}% da base), '
      f'{y[enc_sopro].mean() * 100:.1f}% Anormais; sopro confirmado no exame em '
      f'{sopro_exame[enc_sopro].mean() * 100:.1f}% deles')
print(f'com sopro no exame: {fmt_n(sopro_exame.sum())}; destes, só '
      f'{enc_sopro[sopro_exame].mean() * 100:.1f}% vieram encaminhados por sopro')
exportar(tab_sopro.reset_index(), 'sopro_motivo_x_exame')

                                      consultas  % Anormal
encaminhado por sopro sopro no exame                      
não                   não                  6113        9.9
                      sim                  2341       96.3
sim                   não                   469       11.7
                      sim                  1653       94.8
encaminhados por sopro: 2.122 (20.1% da base), 76.4% Anormais; sopro confirmado no exame em 77.9% deles
com sopro no exame: 3.994; destes, só 41.4% vieram encaminhados por sopro


### 5.3 Como a idade e a frequência cardíaca agem: dependência parcial

In [ ]:
etapa('Parte 5.3 - dependência parcial')

media_log, dp_log = prep_final.media_glob_['IDADE_log'], prep_final.dp_glob_['IDADE_log']
anos = np.linspace(0, 18.5, 75)
z_anos = (np.log1p(anos) - media_log) / dp_log      # a idade entra como z-score de log(1 + idade)
fc_grade = np.linspace(*np.percentile(Xt_tr['FC_z'], [2, 98]), 60)


def dep_parcial(n, coluna, grade):
    r = partial_dependence(finais[n].named_steps['modelo'], Xt_teste[n], [coluna],
                           custom_values={coluna: grade}, kind='average',
                           method='brute', response_method='predict_proba')
    return r['average'][0]


curvas = {n: {'idade': dep_parcial(n, 'IDADE_log_z', z_anos), 'fc': dep_parcial(n, 'FC_z', fc_grade)}
          for n in MODELOS}

cortes = [0, 0.5, 1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 19]
obs_idade = y_treino.groupby(pd.cut(X_treino['IDADE'], cortes, include_lowest=True),
                             observed=True).agg(['mean', 'size'])
meio_idade = [(i.left + i.right) / 2 for i in obs_idade.index]
tem_fc = X_treino['FC'].notna().values              # FC ausentes foram imputadas: ficam de fora
fc_medida = Xt_tr.loc[tem_fc, 'FC_z']
decil_fc = pd.qcut(fc_medida, 10)
obs_fc = y_treino[tem_fc].groupby(decil_fc, observed=True).agg(['mean', 'size'])
meio_fc = fc_medida.groupby(decil_fc, observed=True).median()


[ 210.1s] Parte 5.3 - dependência parcial


## 6. A régua do vazamento: quanto o exame físico explica

O cenário principal exclui os achados do exame físico. Para medir o que essa exclusão custa (e,
portanto, o tamanho do vazamento), os mesmos modelos são avaliados com 47, 37 e 19 colunas. Os
hiperparâmetros são os escolhidos no cenário principal, sem novo ajuste: a comparação é da
informação disponível, não da capacidade de ajuste.

In [ ]:
etapa('Parte 6 - régua do vazamento (47 / 37 / 19 colunas)')

regua = []
for nome_cen, grupos_fora in CENARIOS.items():
    for n in MODELOS:
        modelo = clone(finais[n]).set_params(prep__excluir_grupos=grupos_fora)
        s = cross_validate(modelo, X_treino, y_treino, cv=cv, scoring='roc_auc', n_jobs=-1)['test_score']
        modelo.fit(X_treino, y_treino)
        p = modelo.predict_proba(X_teste)[:, 1]
        n_cols = len(modelo.named_steps['prep'].get_feature_names_out())
        regua.append({'cenário': nome_cen, 'atributos': n_cols, 'modelo': n,
                      'AUC validação cruzada': round(s.mean(), 4), 'desvio': round(s.std(), 4),
                      'AUC teste': round(roc_auc_score(yt, p), 4)})
    print(f'  {nome_cen}: ok', flush=True)
regua = pd.DataFrame(regua)
print(regua.to_string(index=False))
exportar(regua, 'regua_vazamento')

pr = regua[regua['cenário'] == CENARIO_PRINCIPAL].set_index('modelo')
pa = regua[regua['cenário'] == 'Com exame físico (47)'].set_index('modelo')
pt = regua[regua['cenário'] == 'Sem exame físico e sem motivo (19)'].set_index('modelo')
print(f'\n{gb}: {num(pa.loc[gb, "AUC validação cruzada"], 3)} com o exame físico, '
      f'{num(pr.loc[gb, "AUC validação cruzada"], 3)} sem ele '
      f'(diferença de {num(pa.loc[gb, "AUC validação cruzada"] - pr.loc[gb, "AUC validação cruzada"], 3)}); '
      f'{num(pt.loc[gb, "AUC validação cruzada"], 3)} sem o motivo do encaminhamento')


[ 229.8s] Parte 6 - régua do vazamento (47 / 37 / 19 colunas)
  Com exame físico (47): ok
  Sem exame físico (37): ok
  Sem exame físico e sem motivo (19): ok
                           cenário  atributos              modelo  AUC validação cruzada  desvio  AUC teste
             Com exame físico (47)         47 Regressão Logística                 0.9434  0.0045     0.9491
             Com exame físico (47)         47       Random Forest                 0.9504  0.0039     0.9547
             Com exame físico (47)         47   Gradient Boosting                 0.9502  0.0043     0.9555
             Sem exame físico (37)         37 Regressão Logística                 0.7643  0.0075     0.7923
             Sem exame físico (37)         37       Random Forest                 0.7750  0.0115     0.8030
             Sem exame físico (37)         37   Gradient Boosting                 0.7766  0.0133     0.8023
Sem exame físico e sem motivo (19)         19 Regressão Logística                 0.

## 7. Robustez: o perfil dos atendimentos muda ao longo dos anos

A proporção de laudos anormais cai ao longo da série histórica. A validação cruzada sorteia as
dobras ao acaso e, com isso, mistura os anos. Treinar nos anos iniciais e validar nos finais mede
o que acontece quando o modelo é aplicado a atendimentos futuros. O ano é recuperado do
`UCMF_raw.csv` pelo `ID` e serve apenas a esta verificação.

In [ ]:
etapa('Parte 7 - deriva temporal')

raw = pd.read_csv(RAIZ / 'UCMF_raw.csv')
data = pd.to_datetime(raw['Atendimento'], errors='coerce', dayfirst=True, format='mixed')
serial = pd.to_datetime(pd.to_numeric(raw['Atendimento'], errors='coerce'),   # datas gravadas como
                        unit='D', origin='1899-12-30', errors='coerce')       # serial do Excel
ano_atend = data.fillna(serial).dt.year
ano_por_id = pd.Series(ano_atend.values, index=raw['ID'].values)
anos_df = df['ID'].map(ano_por_id)

# Anos com pouquíssimos registros são erro de digitação na data (1967, 2023, 2066) e ficam de fora.
contagem = anos_df.value_counts()
anos_validos = sorted(contagem[contagem >= 100].index)
validos = anos_df.isin(anos_validos)
por_ano = (pd.DataFrame({'ano': anos_df[validos].astype(int), 'anormal': y[validos.to_numpy()]})
           .groupby('ano').agg(consultas=('anormal', 'size'),
                               pct_anormal=('anormal', lambda s: round(s.mean() * 100, 1))))
print(f'anos considerados: {int(min(anos_validos))} a {int(max(anos_validos))}')
print(por_ano.to_string())
exportar(por_ano.reset_index(), 'anormal_por_ano')

CORTE = 2007
i_tr = np.where(validos.to_numpy() & (anos_df <= CORTE).to_numpy())[0]
i_te = np.where(validos.to_numpy() & (anos_df > CORTE).to_numpy())[0]
deriva = []
for n in MODELOS:
    m = clone(finais[n]).fit(df.iloc[i_tr], y.iloc[i_tr])
    p = m.predict_proba(df.iloc[i_te])[:, 1]
    deriva.append({'modelo': n,
                   'AUC teste (dobras ao acaso)': round(roc_auc_score(yt, p_teste[n]), 4),
                   f'AUC treino até {CORTE} -> {CORTE + 1} em diante': round(roc_auc_score(y.iloc[i_te], p), 4)})
deriva = pd.DataFrame(deriva)
deriva.attrs['n'] = (len(i_tr), len(i_te))
print(f'treino {len(i_tr)} consultas (até {CORTE}) | validação {len(i_te)} '
      f'({y.iloc[i_te].mean() * 100:.1f}% Anormal, contra {BASE * 100:.1f}% na base)')
print(deriva.to_string(index=False))
exportar(pd.DataFrame([{'protocolo': 'dobras ao acaso (teste 80/20)', 'consultas de treino': len(X_treino),
                        'consultas de validação': len(X_teste),
                        '% Anormal na validação': round(y_teste.mean() * 100, 1),
                        **{n: round(roc_auc_score(yt, p_teste[n]), 4) for n in MODELOS}},
                       {'protocolo': f'treino até {CORTE}, validação de {CORTE + 1} em diante',
                        'consultas de treino': len(i_tr), 'consultas de validação': len(i_te),
                        '% Anormal na validação': round(y.iloc[i_te].mean() * 100, 1),
                        **{r['modelo']: r[f'AUC treino até {CORTE} -> {CORTE + 1} em diante']
                           for _, r in deriva.iterrows()}}]), 'deriva_temporal')


[ 257.4s] Parte 7 - deriva temporal
anos considerados: 2003 a 2010
      consultas  pct_anormal
ano                         
2003        755         55.8
2004       1172         45.8
2005       1315         47.8
2006       1322         43.6
2007       1508         41.4
2008       1735         42.4
2009       2060         36.8
2010        702         28.6
treino 6072 consultas (até 2007) | validação 4497 (37.7% Anormal, contra 42.4% na base)
             modelo  AUC teste (dobras ao acaso)  AUC treino até 2007 -> 2008 em diante
Regressão Logística                       0.7923                                 0.7714
      Random Forest                       0.8030                                 0.7706
  Gradient Boosting                       0.8023                                 0.7644


## 8. Confirmação independente: validação cruzada aninhada 5x3 na base inteira

O protocolo principal usa um teste de 20% guardado. Uma crítica legítima é que o número depende
do sorteio desse corte. A validação cruzada aninhada responde a isso: cinco dobras externas de
avaliação, e dentro de cada uma três dobras internas que escolhem os hiperparâmetros do zero.
Nenhuma linha participa da escolha e da avaliação, e a estimativa usa as 10.576 consultas.
As grades são reduzidas, para o tempo de execução ficar razoável.

In [ ]:
etapa('Parte 8 - validação cruzada aninhada 5x3')

GRADES_ANINHADA = {
    'Regressão Logística': (LogisticRegression(max_iter=5000), {'modelo__C': [0.01, 0.1, 1, 10]}),
    'Gradient Boosting': (HistGradientBoostingClassifier(early_stopping=False, random_state=RANDOM_STATE),
                          {'modelo__learning_rate': [0.05, 0.1], 'modelo__max_leaf_nodes': [15, 31],
                           'modelo__min_samples_leaf': [20, 50]}),
}
aninhada = []
for n, (base_modelo, grade) in GRADES_ANINHADA.items():
    interna = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE + 1)
    busca = GridSearchCV(pipeline(base_modelo), grade, scoring='roc_auc', cv=interna, n_jobs=-1)
    s = cross_validate(busca, df, y, cv=dobras(), scoring='roc_auc', n_jobs=1)['test_score']
    aninhada.append({'modelo': n, 'protocolo': 'VC aninhada 5x3 (base inteira)',
                     'AUC média': round(s.mean(), 4), 'desvio entre dobras': round(s.std(), 4),
                     'AUC no teste 80/20': round(roc_auc_score(yt, p_teste[n]), 4)})
    print(f'  {n}: {num(s.mean(), 3)} ± {num(s.std(), 3)}', flush=True)
aninhada = pd.DataFrame(aninhada)
print(aninhada.to_string(index=False))
exportar(aninhada, 'vc_aninhada')


[ 261.8s] Parte 8 - validação cruzada aninhada 5x3
  Regressão Logística: 0,771 ± 0,012
  Gradient Boosting: 0,786 ± 0,013
             modelo                      protocolo  AUC média  desvio entre dobras  AUC no teste 80/20
Regressão Logística VC aninhada 5x3 (base inteira)     0.7712               0.0116              0.7923
  Gradient Boosting VC aninhada 5x3 (base inteira)     0.7856               0.0127              0.8023


## 9. Figuras do relatório

In [ ]:
etapa('Parte 9 - figuras')

# Figura 1 - o vazamento: % de exames anormais em cada achado alterado do exame físico
with plt.rc_context(ESTILO_IEEE):
    rotulos = [f'{r} (n = {fmt_n(m.sum())})' for r, m in achados][::-1]
    valores = [y[m].mean() for _, m in achados][::-1]
    pos = np.arange(len(valores))
    fig, ax = plt.subplots(figsize=(COLUNA, 2.6))
    ax.barh(pos, valores, height=0.62, color=COR['Anormal'])
    for p_, v in zip(pos, valores):
        ax.text(v + 0.015, p_, f'{v * 100:.0f}%', va='center', fontsize=7)
    ax.axvline(BASE, color=TINTA_2, lw=0.8, ls='--')
    ax.text(BASE + 0.012, -0.55, f'todas as consultas: {num(BASE * 100)}%', fontsize=6.5,
            color=TINTA_2, va='bottom')
    ax.set_yticks(pos, rotulos)
    ax.set(xlim=(0, 1.1), ylim=(-0.7, len(valores) - 0.4), xlabel='% de exames anormais')
    ax.xaxis.set_major_formatter(PCT)
    ax.grid(axis='y', visible=False)
    salvar(fig, 'fig1_vazamento_exame_fisico')

# Figura 2 - o alvo e os sinais legítimos: % de anormais por faixa etária e por motivo
with plt.rc_context(ESTILO_IEEE):
    # As duas metades vêm das tabelas exportadas em 1.4, para a figura e o texto do relatório não
    # poderem divergir. Categorias com menos de 30 consultas ficam fora: a taxa seria instável.
    por_faixa = pd.DataFrame({'mean': alvo_faixa['pct_anormal'] / 100, 'size': alvo_faixa['consultas']})
    por_motivo = (pd.DataFrame({'mean': alvo_motivo['pct_anormal'] / 100,
                                'size': alvo_motivo['consultas']})
                  .query('size >= 30').sort_values('mean'))

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(PAGINA, 2.3), gridspec_kw={'width_ratios': [1, 1.3]})
    x = np.arange(len(por_faixa))
    a1.bar(x, por_faixa['mean'], width=0.62, color=COR['Anormal'])
    for xi, v in zip(x, por_faixa['mean']):
        a1.text(xi, v + 0.015, f'{v * 100:.0f}%', ha='center', va='bottom', fontsize=7,
                bbox=dict(facecolor='white', edgecolor='none', pad=0.6))
    a1.axhline(BASE, color=TINTA_2, lw=0.8, ls='--')
    a1.text(len(x) - 0.5, BASE + 0.012, f'média: {num(BASE * 100)}%', ha='right', va='bottom',
            fontsize=6.5, color=TINTA_2)
    a1.set_xticks(x, [f'{f}\n({fmt_n(nn)})' for f, nn in zip(por_faixa.index, por_faixa['size'])])
    a1.set(ylim=(0, 0.8), xlabel='Faixa etária em anos (n)', ylabel='% de exames anormais',
           title='(a) Por faixa etária')
    a1.yaxis.set_major_formatter(PCT)
    a1.grid(axis='x', visible=False)

    pos = np.arange(len(por_motivo))
    a2.barh(pos, por_motivo['mean'], height=0.6, color=COR['Anormal'])
    for p_, v in zip(pos, por_motivo['mean']):
        a2.text(v + 0.012, p_, f'{v * 100:.0f}%', va='center', fontsize=7)
    a2.axvline(BASE, color=TINTA_2, lw=0.8, ls='--')
    a2.set_yticks(pos, [f'{m} ({fmt_n(nn)})' for m, nn in zip(por_motivo.index, por_motivo['size'])])
    a2.set(xlim=(0, 1.0), xlabel='% de exames anormais', title='(b) Por motivo do encaminhamento')
    a2.xaxis.set_major_formatter(PCT)
    a2.grid(axis='y', visible=False)
    fig.tight_layout()
    salvar(fig, 'fig2_alvo_idade_motivo')

# Figura 3 - a régua do vazamento: AUC por cenário de atributos
with plt.rc_context(ESTILO_IEEE):
    ordem_cen = list(CENARIOS)
    x = np.arange(len(ordem_cen))
    larg = 0.26
    fig, ax = plt.subplots(figsize=(COLUNA, 2.5))
    for k, n in enumerate(MODELOS):
        v = [regua[(regua['cenário'] == c) & (regua['modelo'] == n)]['AUC validação cruzada'].iloc[0]
             for c in ordem_cen]
        desl = (k - 1) * larg
        ax.bar(x + desl, v, width=larg, color=COR_MODELO[n], label=n)
        for xi, vi in zip(x, v):
            ax.text(xi + desl, vi + 0.008, num(vi, 2), ha='center', va='bottom', fontsize=6)
    ax.axhline(0.5, color=TINTA_2, lw=0.8, ls='--')
    # O eixo começa em 0,45, então as barras cobrem toda a altura: o rótulo precisa de um fundo
    # branco para não se perder dentro delas.
    ax.text(len(x) - 0.55, 0.508, 'sorteio: 0,50', ha='right', va='bottom', fontsize=6.5,
            color=TINTA_2, bbox=dict(facecolor='white', edgecolor='none', pad=0.8))
    # Rótulos curtos e em duas linhas: por extenso, o terceiro cenário invade os vizinhos.
    curto = {'Com exame físico (47)': 'Com exame\nfísico\n(47 atributos)',
             'Sem exame físico (37)': 'Sem exame\nfísico\n(37)',
             'Sem exame físico e sem motivo (19)': 'Sem exame físico\ne sem motivo\n(19)'}
    ax.set_xticks(x, [curto[c] for c in ordem_cen])
    ax.set(ylim=(0.45, 1.03), ylabel='AUC na validação cruzada')
    ax.yaxis.set_major_formatter(virgula(2))
    ax.grid(axis='x', visible=False)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.28), ncol=2, frameon=False)
    salvar(fig, 'fig3_regua_vazamento')

# Figura 4 - curvas ROC no teste
with plt.rc_context(ESTILO_IEEE):
    fig, ax = plt.subplots(figsize=(COLUNA, 3.2))
    ax.plot([0, 1], [0, 1], color=TINTA_2, lw=0.8, ls='--', label='Sorteio / Dummy (0,500)')
    for n in MODELOS:
        fpr, tpr, _ = roc_curve(yt, p_teste[n])
        ax.plot(fpr, tpr, color=COR_MODELO[n], lw=1.2, label=f'{n} ({num(roc_auc_score(yt, p_teste[n]), 3)})')
    s_gb, e_gb = prev_gb[yt == 1].mean(), 1 - prev_gb[yt == 0].mean()
    ax.plot(1 - e_gb, s_gb, 'o', ms=5, color=COR_MODELO[gb], mec='white', mew=1, zorder=4)
    ax.annotate(f'limiar {num(limiar[gb], 2)}\nsens. {s_gb * 100:.0f}%\nespec. {e_gb * 100:.0f}%',
                (1 - e_gb, s_gb), xytext=(-8, 22), textcoords='offset points', ha='right',
                va='center', fontsize=6.5, arrowprops=dict(arrowstyle='-', color=TINTA_2, lw=0.6))
    ax.set(xlim=(0, 1), ylim=(0, 1.01), aspect='equal', xlabel='1 − especificidade',
           ylabel='Sensibilidade')
    ax.xaxis.set_major_formatter(PCT)
    ax.yaxis.set_major_formatter(PCT)
    ax.legend(title='AUC no teste', loc='lower right', frameon=False, fontsize=6.5, title_fontsize=6.5)
    salvar(fig, 'fig4_roc_teste')

# Figura 5 - importância por grupo de atributos
with plt.rc_context(ESTILO_IEEE):
    ordem_fig = list(ordem_grupos)[::-1]
    y0 = np.arange(len(ordem_fig))
    desloc = {'Regressão Logística': -0.24, 'Random Forest': 0.0, 'Gradient Boosting': 0.24}
    fig, ax = plt.subplots(figsize=(COLUNA, 2.5))
    for n in MODELOS:
        ax.errorbar(imp_media.loc[ordem_fig, n], y0 + desloc[n], xerr=imp_dp.loc[ordem_fig, n],
                    fmt='o', ms=3.5, color=COR_MODELO[n], mec='white', mew=0.6, elinewidth=0.8,
                    capsize=0, label=n)
    ax.set_yticks(y0, [f'{g_} ({len(GRUPOS[g_])})' for g_ in ordem_fig])
    ax.axvline(0, color=EIXO, lw=0.6)
    ax.xaxis.set_major_formatter(virgula(2))
    ax.set_xlabel('Queda na AUC do teste ao embaralhar o grupo')
    ax.grid(axis='y', visible=False)
    ax.legend(loc='lower right', frameon=False)
    salvar(fig, 'fig5_importancia_grupos')

# Figura 6 - onde o modelo erra: sensibilidade por motivo do encaminhamento
with plt.rc_context(ESTILO_IEEE):
    em = erros_motivo.sort_values('% Anormal')
    pos = np.arange(len(em))
    fig, ax = plt.subplots(figsize=(COLUNA, 2.6))
    ax.barh(pos + 0.19, em['sensibilidade'], height=0.36, color=COR['Anormal'], label='Sensibilidade')
    ax.barh(pos - 0.19, em['especificidade'], height=0.36, color=COR['Normal'], label='Especificidade')
    for p_, (s_, e_) in enumerate(zip(em['sensibilidade'], em['especificidade'])):
        ax.text(s_ + 0.012, p_ + 0.19, f'{s_ * 100:.0f}%', va='center', fontsize=6)
        ax.text(e_ + 0.012, p_ - 0.19, f'{e_ * 100:.0f}%', va='center', fontsize=6)
    ax.set_yticks(pos, [f'{m}\nn = {fmt_n(r["n"])} · {r["% Anormal"]:.0f}% Anormais'
                        for m, r in em.iterrows()])
    ax.set(xlim=(0, 1.12), xlabel=f'Taxa no teste ({gb}, limiar {num(limiar[gb], 2)})')
    ax.xaxis.set_major_formatter(PCT)
    ax.grid(axis='y', visible=False)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=2, frameon=False)
    salvar(fig, 'fig6_erros_por_motivo')

# Figura 7 - dependência parcial: idade e frequência cardíaca
with plt.rc_context(ESTILO_IEEE):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(PAGINA, 2.4), sharey=True)
    ax1.plot(meio_idade, obs_idade['mean'], 'o', ms=3.5, color=TINTA_2, alpha=0.7)
    ax2.plot(meio_fc, obs_fc['mean'], 'o', ms=3.5, color=TINTA_2, alpha=0.7,
             label='Proporção observada (treino)')
    for n in MODELOS:
        ax1.plot(anos, curvas[n]['idade'], color=COR_MODELO[n], lw=1.2)
        ax2.plot(fc_grade, curvas[n]['fc'], color=COR_MODELO[n], lw=1.2, label=n)
    ax1.set(xlim=(0, 19), ylim=(0, 0.8), xlabel='Idade (anos)',
            ylabel='Probabilidade prevista\nde exame anormal', title='(a) Idade')
    ax1.set_xticks(range(0, 20, 2))
    ax2.axvline(0, color=EIXO, lw=0.6, zorder=1)
    ax2.set(xlabel='FC em desvios-padrão da média da faixa etária', title='(b) Frequência cardíaca')
    ax2.xaxis.set_major_formatter(virgula(0))
    ax1.yaxis.set_major_formatter(PCT)
    fig.legend(*ax2.get_legend_handles_labels(), loc='lower center', ncol=4, frameon=False,
               bbox_to_anchor=(0.5, -0.08))
    fig.tight_layout()
    salvar(fig, 'fig7_dependencia_parcial')


[ 288.2s] Parte 9 - figuras


## 10. Resumo dos dados, para a seção de Dados do relatório

In [ ]:
resumo = pd.DataFrame([
    ('consultas na base bruta', fmt_n(len(raw))),
    ('variáveis na base bruta', str(raw.shape[1])),
    ('consultas na base limpa (A1)', fmt_n(len(df))),
    ('exames anormais na base limpa', f'{fmt_n(y.sum())} ({num(BASE * 100)}%)'),
    ('atributos codificados (A1)', str(len(cols_a1))),
    ('atributos de entrada dos modelos (sem o exame físico)', str(len(cols_p))),
    ('consultas de treino / teste', f'{fmt_n(len(X_treino))} / {fmt_n(len(X_teste))}'),
    ('% Anormal no treino / teste', f'{num(y_treino.mean() * 100)}% / {num(y_teste.mean() * 100)}%'),
], columns=['item', 'valor'])
print(resumo.to_string(index=False))
exportar(resumo, 'resumo_dados')

etapa(f'fim - {len(list(TAB.glob("*.csv")))} tabelas e {len(list(FIG.glob("*.png")))} figuras em {PASTA.name}')

                                                 item         valor
                              consultas na base bruta        12.873
                              variáveis na base bruta            21
                         consultas na base limpa (A1)        10.576
                        exames anormais na base limpa 4.485 (42,4%)
                           atributos codificados (A1)            47
atributos de entrada dos modelos (sem o exame físico)            37
                          consultas de treino / teste 8.460 / 2.116
                          % Anormal no treino / teste 42,4% / 42,4%

[ 290.5s] fim - 28 tabelas e 7 figuras em Modelagem_A2
